In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.fft import fft
from scipy.stats import entropy
import joblib
from obspy.core import read

# Path manifest hasil cleaning
manifest_path = r"E:\Skripsi\DEC\dataset\ae-supervised-dataset\cleaned-baru-XGB\manifest.csv"
splits_root = r"E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB"
os.makedirs(splits_root, exist_ok=True)

# Load manifest
df = pd.read_csv(manifest_path)
print("Total data:", len(df))
print("Label distribusi:\n", df["label"].value_counts())


Total data: 2808
Label distribusi:
 label
MP          1430
ROCKFALL    1252
VTB          126
Name: count, dtype: int64


In [2]:
def extract_handcrafted_features(waveform):
    features = {}
    
    # Basic Statistical Features
    features['mean'] = np.mean(waveform)
    features['median'] = np.median(waveform)
    features['std'] = np.std(waveform)
    features['skewness'] = stats.skew(waveform)
    features['kurtosis'] = stats.kurtosis(waveform)
    
    # Entropy Features
    hist, _ = np.histogram(waveform, bins=50, density=True)
    hist = hist[hist > 0]
    features['shannon_entropy'] = entropy(hist)
    features['renyi_entropy'] = -np.log(np.sum(hist**2))
    
    # Rate of Attack and Decay
    peak_idx = np.argmax(np.abs(waveform))
    features['rate_of_attack'] = peak_idx / len(waveform) if peak_idx > 0 else 0
    features['rate_of_decay'] = (len(waveform) - peak_idx) / len(waveform) if peak_idx < len(waveform) - 1 else 0
    
    # Spectral Feature (RMS Bandwidth)
    fft_vals = np.abs(fft(waveform))
    freqs = np.fft.fftfreq(len(waveform))
    positive_freqs = freqs[freqs > 0]
    positive_fft = fft_vals[freqs > 0]
    
    if len(positive_fft) > 0:
        spectral_centroid = np.sum(positive_freqs * positive_fft) / np.sum(positive_fft)
        spectral_rms = np.sqrt(np.sum((positive_freqs - spectral_centroid)**2 * positive_fft) / np.sum(positive_fft))
        features['rms_bandwidth'] = spectral_rms
    else:
        features['rms_bandwidth'] = 0
    
    return features

In [4]:
def process_split_and_save(split_df, split_name):
    feature_rows = []

    for idx, row in split_df.iterrows():
        file_path = row['path']
        label = row['label']

        try:
            st = read(file_path)
            waveform = st[0].data
            features = extract_handcrafted_features(waveform)
            features['path'] = file_path
            features['label'] = label
            feature_rows.append(features)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # Save RAW version (tanpa scaling)
    feature_df = pd.DataFrame(feature_rows)
    raw_path = os.path.join(splits_root, f"{split_name}.csv")
    feature_df.to_csv(raw_path, index=False)
    print(f"Saved RAW CSV: {raw_path}")

    return feature_df

In [5]:
train_val_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=42
)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.20, stratify=train_val_df["label"], random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# Ekstraksi fitur & simpan versi RAW
train_raw = process_split_and_save(train_df, "train")
val_raw = process_split_and_save(val_df, "val")
test_raw = process_split_and_save(test_df, "test")


Train: 1796
Validation: 450
Test: 562
Saved RAW CSV: E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB\train.csv
Saved RAW CSV: E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB\val.csv
Saved RAW CSV: E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB\test.csv


In [6]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_raw.drop(columns=["path", "label"]))
val_scaled = scaler.transform(val_raw.drop(columns=["path", "label"]))
test_scaled = scaler.transform(test_raw.drop(columns=["path", "label"]))

# Tambah kolom kembali
train_scaled_df = pd.DataFrame(train_scaled, columns=train_raw.columns[:-2])
val_scaled_df = pd.DataFrame(val_scaled, columns=val_raw.columns[:-2])
test_scaled_df = pd.DataFrame(test_scaled, columns=test_raw.columns[:-2])

# Tambahkan path + label
for df_scaled, df_raw in [
    (train_scaled_df, train_raw), 
    (val_scaled_df, val_raw),
    (test_scaled_df, test_raw)
]:
    df_scaled["path"] = df_raw["path"].values
    df_scaled["label"] = df_raw["label"].values

# Simpan versi scaled
train_scaled_df.to_csv(os.path.join(splits_root, "train_scaled.csv"), index=False)
val_scaled_df.to_csv(os.path.join(splits_root, "val_scaled.csv"), index=False)
test_scaled_df.to_csv(os.path.join(splits_root, "test_scaled.csv"), index=False)

# Simpan scaler untuk inference nanti
joblib.dump(scaler, os.path.join(splits_root, "scaler.pkl"))

print("\n✔ Semua CSV versi raw & scaled tersimpan.")
print("✔ Scaler disimpan sebagai scaler.pkl")



✔ Semua CSV versi raw & scaled tersimpan.
✔ Scaler disimpan sebagai scaler.pkl


In [7]:
print("\nDistribusi Train:")
print(train_df["label"].value_counts())
print("\nDistribusi Val:")
print(val_df["label"].value_counts())
print("\nDistribusi Test:")
print(test_df["label"].value_counts())


Distribusi Train:
label
MP          915
ROCKFALL    800
VTB          81
Name: count, dtype: int64

Distribusi Val:
label
MP          229
ROCKFALL    201
VTB          20
Name: count, dtype: int64

Distribusi Test:
label
MP          286
ROCKFALL    251
VTB          25
Name: count, dtype: int64


In [8]:
import pandas as pd

df = pd.read_csv(r"E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB\train.csv")
print(df.drop(columns=["path", "label"]).describe().T)


                  count         mean          std         min          25%  \
mean             1796.0    -0.210601    15.576879  -91.942210    -5.535844   
median           1796.0     0.917132    96.445596 -649.235700   -36.019838   
std              1796.0  7504.159017  6183.565940  727.621030  2662.595925   
skewness         1796.0    -0.000552     0.140861   -0.989713    -0.080264   
kurtosis         1796.0     2.528510     2.435675   -0.186349     1.182080   
shannon_entropy  1796.0     3.005670     0.269881    1.326271     2.909566   
renyi_entropy    1796.0    16.464670     1.614376   12.650437    15.108497   
rate_of_attack   1796.0     0.498332     0.265663    0.000667     0.275333   
rate_of_decay    1796.0     0.501667     0.265663    0.000000     0.272917   
rms_bandwidth    1796.0     0.055099     0.023867    0.032708     0.041411   

                         50%           75%           max  
mean               -0.048375      4.819233    180.049090  
median             -1.0

In [ ]:
import pandas as pd

df = pd.read_csv(r"E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB\train_scaled.csv")
print(df.drop(columns=["path", "label"]).describe().T)


                  count          mean       std       min       25%       50%  \
mean             1796.0 -1.087969e-17  1.000279 -5.890599 -0.341964  0.010417   
median           1796.0  5.934377e-18  1.000279 -6.743013 -0.383089 -0.020529   
std              1796.0  4.747502e-17  1.000279 -1.096200 -0.783191 -0.296457   
skewness         1796.0 -7.912503e-18  1.000279 -7.024226 -0.566051 -0.029021   
kurtosis         1796.0  9.692816e-17  1.000279 -1.114933 -0.552949 -0.257610   
shannon_entropy  1796.0 -7.833378e-16  1.000279 -6.224474 -0.356196  0.179753   
renyi_entropy    1796.0 -2.195719e-16  1.000279 -2.363324 -0.840294 -0.003278   
rate_of_attack   1796.0 -1.226438e-16  1.000279 -1.873820 -0.839640 -0.286153   
rate_of_decay    1796.0 -1.701188e-16  1.000279 -1.888884 -0.861295  0.286153   
rms_bandwidth    1796.0  2.284735e-16  1.000279 -0.938399 -0.573644 -0.337215   

                      75%        max  
mean             0.322994  11.575482  
median           0.322672   7.

In [ ]:
df = pd.read_csv(r"E:\Skripsi\DEC\dataset\ae-supervised-dataset\splits-baru-XGB\train_scaled.csv")
print(df.drop(columns=["path", "label"]).describe().T[['mean','std']])


                         mean       std
mean            -1.087969e-17  1.000279
median           5.934377e-18  1.000279
std              4.747502e-17  1.000279
skewness        -7.912503e-18  1.000279
kurtosis         9.692816e-17  1.000279
shannon_entropy -7.833378e-16  1.000279
renyi_entropy   -2.195719e-16  1.000279
rate_of_attack  -1.226438e-16  1.000279
rate_of_decay   -1.701188e-16  1.000279
rms_bandwidth    2.284735e-16  1.000279
